In [1]:
import requests  
import json      
import datetime  
import os
from pyspark.sql import SparkSession

# Inicializa a sessão PySpark
spark = SparkSession.builder.appName("SteamGames").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")  # Ajuste o nível de log para ERROR

24/11/15 19:29:30 WARN Utils: Your hostname, fececa-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
24/11/15 19:29:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/11/15 19:29:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

In [3]:
# Verificar o ambiente
if "dev" in current_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [4]:
with open(dir_path + f'/{env}-env/projeto_steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [5]:
# Função para obter os detalhes dos jogos
def get_game_wish(steam_id):
    url = f'https://store.steampowered.com/wishlist/profiles/{steam_id}/wishlistdata/'
    
    response = requests.get(url)
    
    if response.status_code == 200:
        wishlist = response.json()
        
        wishlist_data = []
    
        for app_id, details in wishlist.items():
            store_url = f'https://store.steampowered.com/api/appdetails?appids={app_id}&cc=br&l=pt'
            store_response = requests.get(store_url)
    
            if store_response.status_code == 200:
                store_data = store_response.json()
                if store_data[str(app_id)]['success']:
                    game_data = store_data[str(app_id)]['data']
                    
                    price_info = game_data.get('price_overview', {})
                    price = price_info.get('final_formatted', 'N/A')
                    pc_requirements = game_data.get('pc_requirements', {})
                    minimum_requirements = pc_requirements.get('minimum', 'N/A')
                    recommended_requirements = pc_requirements.get('recommended', 'N/A')
                    description = game_data.get('short_description', 'N/A')
                    header_image = game_data.get('header_image', 'N/A')
                    steam_url = f"https://store.steampowered.com/app/{app_id}"
                    
                    wishlist_data.append({
                        'appid': app_id,
                        'name': details.get('name', 'N/A'),
                        'price': price,
                        'release_date': details.get('release_date', 'N/A'),
                        'discount_pct': details.get('discount_pct', 0),
                        'minimum': minimum_requirements,
                        'recommended': recommended_requirements,
                        'short_description': description,
                        'header_image': header_image,
                        'link': steam_url
                    })
                else:
                    print(f"Erro ao obter detalhes do jogo {app_id}")
            else:
                print(f"Erro ao obter detalhes do jogo {app_id}")

    return wishlist_data

In [6]:
# Substitua suas credenciais aqui
STEAM_ID = config["STEAM_ID"]

# Obtenha os detalhes dos jogos
wish = get_game_wish(STEAM_ID)

In [7]:
# Crie um dataframe PySpark a partir dos dados processados
df = spark.createDataFrame(wish)

# Mostre o dataframe
df.show()

+-------+------------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+------------+--------------------+
|  appid|discount_pct|        header_image|                link|             minimum|                name|    price|         recommended|release_date|   short_description|
+-------+------------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+------------+--------------------+
|  15100|           0|https://shared.ak...|https://store.ste...|<strong>Minimum: ...|Assassin's Creed™...| R$ 59,99|                 N/A|  1207785600|Assassin's Creed™...|
|  33230|           0|https://shared.ak...|https://store.ste...|<strong>Minimum:<...|  Assassin's Creed 2| R$ 59,99|<strong>Recommend...|  1268157600|An epic story of ...|
|  48190|           0|https://shared.ak...|https://store.ste...|<strong>Minimum</...|Assassin’s Creed®...| R$ 59,99|<strong>Recommend...|  1

In [8]:
# Salve o dataframe como um único arquivo JSON
current_date_str = datetime.datetime.now().strftime("%Y-%m-%d")
base_path = dir_path + f'/{env}-env/datalake/steam/wishlist/inbound'
os.makedirs(base_path, exist_ok=True)
temp_path = os.path.join(base_path, "temp_json")

# Coalesce para garantir que seja salvo em um único arquivo
df.coalesce(1).write.json(temp_path, mode="overwrite")

# Renomeie o arquivo salvo
for file_name in os.listdir(temp_path):
    if file_name.endswith(".json"):
        os.rename(os.path.join(temp_path, file_name), os.path.join(base_path, f"{current_date_str}.json"))

# Remova todos os arquivos na pasta temporária
for file_name in os.listdir(temp_path):
    os.remove(os.path.join(temp_path, file_name))

# Remova o diretório temporário
os.rmdir(temp_path)